<a href="https://colab.research.google.com/github/JonnaBauer/DeepLearningProject/blob/main/Modernbert_peft_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y torchvision
!pip install -q transformers datasets peft evaluate scikit-learn seqeval

In [ ]:
import time, numpy as np, torch
from datasets import load_dataset
from transformers import (AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments, Trainer)
from peft import get_peft_model, LoraConfig, TaskType
import evaluate

print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
DATASET_NAME = "glue/qnli"

if DATASET_NAME == "glue/qnli":
    dataset = load_dataset("nyu-mll/glue", "qnli")
    text_col = "question"
else:
    dataset = load_dataset("stanfordnlp/sst2")
    text_col = "sentence"

MODEL_NAME = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch[text_col], truncation=True,
                     padding="max_length", max_length=128)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch",
    columns=["input_ids", "attention_mask", "labels"])
print("Dataset ready:", DATASET_NAME)

In [ ]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

In [ ]:
# ── change only these values ────────────────────────────────
DATASET_NAME  = "glue/qnli"  # or "glue/qnli"
USE_LORA      = True   # True = LoRA
USE_IA3       = False   # True = IA³
LORA_RANK     = 8       # only used if USE_LORA=True
NUM_EPOCHS    = 3
TRAIN_SAMPLES = None
# ────────────────────────────────────────────────────────────

In [ ]:
# number of labels per dataset
num_labels = 3 if "qnli" in DATASET_NAME else 2
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels)

if USE_LORA:
    from peft import LoraConfig, TaskType, get_peft_model
    cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=LORA_RANK,
        lora_alpha=LORA_RANK * 2,
        target_modules=["Wqkv", "dense"],
        lora_dropout=0.1,
        bias="none"
    )
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
elif USE_IA3:
    from peft import IA3Config, get_peft_model
    cfg = IA3Config(
        task_type=TaskType.SEQ_CLS,
        target_modules=["Wqkv", "dense"],
        feedforward_modules=["dense"]
    )
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
else:
    n = sum(p.numel() for p in model.parameters())
    print(f"Full fine-tuning — trainable params: {n:,}")

In [ ]:
train_ds = tokenized["train"]
eval_ds  = tokenized["validation"]

if TRAIN_SAMPLES:
    train_ds = train_ds.shuffle(seed=42).select(range(TRAIN_SAMPLES))
    print(f"Using {TRAIN_SAMPLES} training samples")
else:
    print(f"Using full training set ({len(train_ds)} samples)")

In [ ]:
args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    seed=42,
    report_to="none"
)

trainer = Trainer(model=model, args=args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    compute_metrics=compute_metrics)

start = time.time()
trainer.train()
elapsed = time.time() - start
print(f"Training time: {elapsed/60:.1f} min")

In [ ]:
results = trainer.evaluate()
trainable = sum(p.numel() for p in model.parameters()
                if p.requires_grad)

print("=" * 40)
print(f"USE_LORA:         {USE_LORA}")
print(f"LORA_RANK:        {LORA_RANK if USE_LORA else 'N/A'}")
print(f"NUM_EPOCHS:       {NUM_EPOCHS}")
print(f"TRAIN_SAMPLES:    {TRAIN_SAMPLES or 'full'}")
print(f"Val accuracy:     {results['eval_accuracy']:.4f}")
print(f"Trainable params: {trainable:,}")
print(f"Training time:    {elapsed/60:.1f} min")
print("=" * 40)